# Simple Block Comparison Evaluation

This notebook compares **manual/reference JSON blocks** with **Llama/AI detected JSON blocks**.

It answers only three questions for each manual block:

1. **Did Llama find similar text?**
2. **Did Llama give the correct label?**
3. **Was it in the correct order?**

It also creates:

- overall summary per sample
- block-level comparison report
- per-label accuracy
- label confusion table
- unmatched/extra Llama blocks


## Step 1 — Import libraries

In [1]:
from pathlib import Path
import json
import re
from difflib import SequenceMatcher

import pandas as pd

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 200)


## Step 2 — Set project folders

Update these folders only if your files are somewhere else.

Expected file names:

- `sample1_reference_blocks.json`
- `sample1_llama_blocks.json`

The code also handles names like `sample1_reference__blocks.json`.


In [2]:
# Find project root
cwd = Path.cwd()

if (cwd / 'data').exists():
    project_root = cwd
elif (cwd.parent / 'data').exists():
    project_root = cwd.parent
else:
    # Fallback for uploaded files in this environment
    project_root = Path('/mnt/data')

# Main folders
reference_dir = project_root / 'data' / 'reference_structure_blocks'
llama_dir = project_root / 'data' / 'llama_structured_blocks'

# If folders do not exist, use /mnt/data for uploaded sample files
if not reference_dir.exists():
    reference_dir = Path('/mnt/data')
if not llama_dir.exists():
    llama_dir = Path('/mnt/data')

output_dir = project_root / 'data' / 'block_comparison_reports_simple'
if project_root == Path('/mnt/data'):
    output_dir = Path('/mnt/data/block_comparison_reports_simple')

output_dir.mkdir(parents=True, exist_ok=True)

print('Project root:', project_root)
print('Reference folder:', reference_dir)
print('Llama folder:', llama_dir)
print('Output folder:', output_dir)


Project root: c:\Users\Nahid\dmpbridge
Reference folder: c:\Users\Nahid\dmpbridge\data\reference_structure_blocks
Llama folder: c:\Users\Nahid\dmpbridge\data\llama_structured_blocks
Output folder: c:\Users\Nahid\dmpbridge\data\block_comparison_reports_simple


## Step 3 — Find JSON files

In [4]:
reference_files = sorted(reference_dir.glob('*reference*blocks*.json'))
llama_files = sorted(llama_dir.glob('*llama*blocks*.json'))

print('Reference files:')
for f in reference_files:
    print(' -', f.name)

print('Llama files:')
for f in llama_files:
    print(' -', f.name)


Reference files:
 - sample10_reference_blocks.json
 - sample1_reference_blocks.json
 - sample2_reference_blocks.json
 - sample3_reference_blocks.json
 - sample4_reference_blocks.json
 - sample5_reference_blocks.json
 - sample6_reference_blocks.json
 - sample7_reference_blocks.json
 - sample8_reference_blocks.json
 - sample9_reference_blocks.json
Llama files:
 - sample10_llama_blocks.json
 - sample1_llama_blocks.json
 - sample2_llama_blocks.json
 - sample3_llama_blocks.json
 - sample4_llama_blocks.json
 - sample5_llama_blocks.json
 - sample6_llama_blocks.json
 - sample7_llama_blocks.json
 - sample8_llama_blocks.json
 - sample9_llama_blocks.json


## Step 4 — Helper functions

These functions keep the comparison simple:

- clean text before matching
- extract sample ID from file name
- compute text similarity
- load JSON blocks


In [6]:
def normalize_text(text):
    """Clean text so small formatting differences do not hurt matching."""
    if text is None:
        return ''
    text = str(text)
    text = text.replace(' ', ' ').replace('	', ' ')
    text = text.replace('“', '"').replace('”', '"').replace('’', "'")
    text = text.replace('–', '-').replace('—', '-')
    text = re.sub(r'\s+', ' ', text)
    return text.strip().lower()


def token_set_similarity(a, b):
    """Word overlap similarity between two text blocks."""
    a_tokens = set(normalize_text(a).split())
    b_tokens = set(normalize_text(b).split())
    if not a_tokens or not b_tokens:
        return 0.0
    return len(a_tokens & b_tokens) / len(a_tokens | b_tokens)


def sequence_similarity(a, b):
    """Character/order similarity between two text blocks."""
    return SequenceMatcher(None, normalize_text(a), normalize_text(b)).ratio()


def combined_similarity(a, b):
    """Final text similarity score: 60% sequence + 40% word overlap."""
    return (0.60 * sequence_similarity(a, b)) + (0.40 * token_set_similarity(a, b))


def sample_id_from_filename(path):
    """Extract sample ID from a file name."""
    name = path.stem.lower()
    name = name.replace('_reference__blocks', '')
    name = name.replace('_reference_blocks', '')
    name = name.replace('_llama_blocks', '')
    name = name.replace('-reference-blocks', '')
    name = name.replace('-llama-blocks', '')
    return name


def load_blocks(json_path):
    """Load blocks from JSON and keep only index, label, text."""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Some JSON files may store blocks inside a dictionary
    if isinstance(data, dict):
        for key in ['blocks', 'structured_blocks', 'data']:
            if key in data and isinstance(data[key], list):
                data = data[key]
                break

    blocks = []
    for i, block in enumerate(data, start=1):
        if not isinstance(block, dict):
            continue
        label = str(block.get('label', '')).strip()
        text = str(block.get('text', '')).strip()
        if not text:
            continue
        blocks.append({
            'idx': i,
            'label': label,
            'text': text,
            'norm_text': normalize_text(text)
        })
    return blocks


## Step 5 — Set simple evaluation rules

A block is counted as **text found** when similarity is at least `SIMILARITY_THRESHOLD`.

A block is counted as **order correct** when the reference index and Llama index are within `ORDER_TOLERANCE` positions.


In [8]:
SIMILARITY_THRESHOLD = 0.75
ORDER_TOLERANCE = 1

print('Similarity threshold:', SIMILARITY_THRESHOLD)
print('Order tolerance:', ORDER_TOLERANCE)


Similarity threshold: 0.75
Order tolerance: 1


## Step 6 — Compare one sample

In [9]:
def make_error_type(similar_text_found, label_correct, order_correct):
    """Simple explanation for each reference block."""
    if not similar_text_found:
        return 'text_not_found'
    if similar_text_found and not label_correct:
        return 'label_wrong'
    if similar_text_found and label_correct and not order_correct:
        return 'order_wrong'
    return 'correct'


def compare_sample(sample_id, reference_path, llama_path,
                   similarity_threshold=SIMILARITY_THRESHOLD,
                   order_tolerance=ORDER_TOLERANCE):
    """Compare one reference JSON file with one Llama JSON file."""

    reference_blocks = load_blocks(reference_path)
    llama_blocks = load_blocks(llama_path)

    used_llama_indices = set()
    rows = []

    for ref in reference_blocks:
        best = None
        best_score = -1

        # Find best unused Llama block by text similarity
        for llama in llama_blocks:
            if llama['idx'] in used_llama_indices:
                continue

            score = combined_similarity(ref['text'], llama['text'])
            if score > best_score:
                best_score = score
                best = llama

        similar_text_found = best is not None and best_score >= similarity_threshold

        if similar_text_found:
            used_llama_indices.add(best['idx'])

            label_correct = ref['label'] == best['label']
            order_difference = abs(ref['idx'] - best['idx'])
            order_correct = order_difference <= order_tolerance

            llama_idx = best['idx']
            llama_label = best['label']
            llama_text = best['text']
        else:
            label_correct = False
            order_difference = None
            order_correct = False
            llama_idx = None
            llama_label = None
            llama_text = None

        rows.append({
            'sample_id': sample_id,
            'ref_idx': ref['idx'],
            'ref_label': ref['label'],
            'ref_text': ref['text'],
            'llama_idx': llama_idx,
            'llama_label': llama_label,
            'llama_text': llama_text,
            'similarity_score': round(best_score, 3) if best_score >= 0 else None,
            'similar_text_found': similar_text_found,
            'label_correct': label_correct,
            'order_difference': order_difference,
            'order_correct': order_correct,
            'error_type': make_error_type(similar_text_found, label_correct, order_correct)
        })

    # Extra Llama blocks: detected by Llama but not matched to any reference block
    extra_rows = []
    for llama in llama_blocks:
        if llama['idx'] not in used_llama_indices:
            extra_rows.append({
                'sample_id': sample_id,
                'llama_idx': llama['idx'],
                'llama_label': llama['label'],
                'llama_text': llama['text'],
                'issue': 'extra_or_unmatched_llama_block'
            })

    block_report = pd.DataFrame(rows)
    extra_report = pd.DataFrame(extra_rows)

    return block_report, extra_report, reference_blocks, llama_blocks


## Step 7 — Match reference files with Llama files

In [11]:
reference_map = {sample_id_from_filename(f): f for f in reference_files}
llama_map = {sample_id_from_filename(f): f for f in llama_files}

common_sample_ids = sorted(set(reference_map) & set(llama_map))

print('Matched sample IDs:')
for sid in common_sample_ids:
    print(' -', sid)

missing_llama = sorted(set(reference_map) - set(llama_map))
missing_reference = sorted(set(llama_map) - set(reference_map))

if missing_llama: print('Reference files without matching Llama file:', missing_llama)
if missing_reference:
    print('Llama files without matching reference file:', missing_reference)


Matched sample IDs:
 - sample1
 - sample10
 - sample2
 - sample3
 - sample4
 - sample5
 - sample6
 - sample7
 - sample8
 - sample9


## Step 8 — Run comparison for all samples

In [12]:
all_block_reports = []
all_extra_reports = []
summary_rows = []

for sample_id in common_sample_ids:
    block_report, extra_report, reference_blocks, llama_blocks = compare_sample(
        sample_id,
        reference_map[sample_id],
        llama_map[sample_id]
    )

    all_block_reports.append(block_report)
    all_extra_reports.append(extra_report)

    total_ref = len(reference_blocks)
    total_llama = len(llama_blocks)

    text_found = int(block_report['similar_text_found'].sum())
    label_correct = int(block_report['label_correct'].sum())
    order_correct = int(block_report['order_correct'].sum())
    wrong_or_missing = int((block_report['error_type'] != 'correct').sum())

    summary_rows.append({
        'sample_id': sample_id,
        'reference_blocks': total_ref,
        'llama_blocks': total_llama,
        'text_found': text_found,
        'label_correct': label_correct,
        'order_correct': order_correct,
        'wrong_or_missing': wrong_or_missing,
        'extra_llama_blocks': len(extra_report),
        'text_match_rate': round(text_found / total_ref, 3) if total_ref else 0,
        'label_accuracy': round(label_correct / total_ref, 3) if total_ref else 0,
        'order_accuracy': round(order_correct / total_ref, 3) if total_ref else 0
    })

block_level_report = pd.concat(all_block_reports, ignore_index=True) if all_block_reports else pd.DataFrame()
extra_llama_report = pd.concat(all_extra_reports, ignore_index=True) if all_extra_reports else pd.DataFrame()
summary_report = pd.DataFrame(summary_rows)

summary_report


,sample_id,reference_blocks,llama_blocks,text_found,label_correct,order_correct,wrong_or_missing,extra_llama_blocks,text_match_rate,label_accuracy,order_accuracy
0,sample1,26,28,26,23,24,5,2,1.000,0.885,0.923
1,sample10,13,13,13,13,13,0,0,1.000,1.000,1.000
2,sample2,12,32,5,5,2,10,27,0.417,0.417,0.167
3,sample3,14,9,7,7,1,13,2,0.500,0.500,0.071
4,sample4,2,26,1,1,1,1,25,0.500,0.500,0.500
5,sample5,13,25,9,9,1,12,16,0.692,0.692,0.077
6,sample6,11,11,3,3,3,8,8,0.273,0.273,0.273
7,sample7,2,2,2,2,2,0,0,1.000,1.000,1.000
8,sample8,13,13,13,13,13,0,0,1.000,1.000,1.000
9,sample9,11,14,11,10,7,4,3,1.000,0.909,0.636


## Step 9 — Block-level report

This is the most important table. Each row is one manual/reference block.

Look at:

- `similar_text_found`
- `label_correct`
- `order_correct`
- `error_type`


In [13]:
display_cols = [
    'sample_id', 'ref_idx', 'ref_label', 'llama_idx', 'llama_label',
    'similarity_score', 'similar_text_found', 'label_correct',
    'order_difference', 'order_correct', 'error_type', 'ref_text', 'llama_text'
]

block_level_report[display_cols]


,sample_id,ref_idx,ref_label,llama_idx,llama_label,similarity_score,similar_text_found,label_correct,order_difference,order_correct,error_type,ref_text,llama_text
0,sample1,1,document_title,1.0,document_title,1.000,True,True,0.0,True,correct,DATA MANAGEMENT AND SHARING PLAN,DATA MANAGEMENT AND SHARING PLAN
1,sample1,2,section,2.0,section,1.000,True,True,0.0,True,correct,Element 1: Data Type:,Element 1: Data Type:
2,sample1,3,subsection,3.0,subsection,1.000,True,True,0.0,True,correct,A. Types and amount of scientific data expected to be generated in the project:,A. Types and amount of scientific data expected to be generated in the project:
3,sample1,4,content,4.0,content,0.981,True,True,0.0,True,correct,"This secondary data analysis project will analyze deidentified data from 48,218 participants from eight studies and ...","This secondary data analysis project will analyze deidentified data from 48,218 participants from eight studies and ..."
4,sample1,5,subsection,5.0,section,1.000,True,False,0.0,True,label_wrong,"B. Scientific data that will be preserved and shared, and the rationale for doing so:","B. Scientific data that will be preserved and shared, and the rationale for doing so:"
5,sample1,6,content,6.0,content,0.959,True,True,0.0,True,correct,"As this is a secondary data analysis project, we will only be able to publicly share in the UC San Diego Library Rep...","As this is a secondary data analysis project, we will only be able to publicly share in the UC San Diego Library Rep..."
6,sample1,7,subsection,7.0,section,1.000,True,False,0.0,True,label_wrong,"C. Metadata, other relevant data, and associated documentation:","C. Metadata, other relevant data, and associated documentation:"
7,sample1,8,content,8.0,content,1.000,True,True,0.0,True,correct,"In addition to the data described above, code and models will be included in the repository and in on our project’s ...","In addition to the data described above, code and models will be included in the repository and in on our project’s ..."
8,sample1,9,section,9.0,section,1.000,True,True,0.0,True,correct,"Element 2: Related Tools, Software and/or Code:","Element 2: Related Tools, Software and/or Code:"
9,sample1,10,content,10.0,content,1.000,True,True,0.0,True,correct,Data will be analyzed with custom code by our statistical and computer science team. ActiGraph data will be processe...,Data will be analyzed with custom code by our statistical and computer science team. ActiGraph data will be processe...


## Step 10 — Show only problem blocks

This table makes it easy to see **which blocks were wrong**.


In [14]:
problem_blocks = block_level_report[block_level_report['error_type'] != 'correct'].copy()
problem_blocks[display_cols]


,sample_id,ref_idx,ref_label,llama_idx,llama_label,similarity_score,similar_text_found,label_correct,order_difference,order_correct,error_type,ref_text,llama_text
4,sample1,5,subsection,5.0,section,1.000,True,False,0.0,True,label_wrong,"B. Scientific data that will be preserved and shared, and the rationale for doing so:","B. Scientific data that will be preserved and shared, and the rationale for doing so:"
6,sample1,7,subsection,7.0,section,1.000,True,False,0.0,True,label_wrong,"C. Metadata, other relevant data, and associated documentation:","C. Metadata, other relevant data, and associated documentation:"
22,sample1,23,subsection,23.0,section,1.000,True,False,0.0,True,label_wrong,B. Whether access to scientific data will be controlled:,B. Whether access to scientific data will be controlled:
24,sample1,25,section,27.0,section,1.000,True,True,2.0,False,order_wrong,Element 6: Oversight of Data Management and Sharing:,Element 6: Oversight of Data Management and Sharing:
25,sample1,26,content,28.0,content,1.000,True,True,2.0,False,order_wrong,"The PI and Trial & IRB Coordinator will provide updates and changes to the plan in the annual RPPR. The PI, Trial & ...","The PI and Trial & IRB Coordinator will provide updates and changes to the plan in the annual RPPR. The PI, Trial & ..."
41,sample2,3,subsection,NaN,None,0.671,False,False,NaN,False,text_not_found,Data management plans should describe whether and how data generated in the course of the proposed research will be ...,None
42,sample2,4,content,NaN,None,0.570,False,False,NaN,False,text_not_found,"Roles & Responsibilities. For the proposed research, Director Samuel Stupp with help from the Executive Director of ...",None
43,sample2,5,section,15.0,section,1.000,True,True,10.0,False,order_wrong,2. Data used in publications,2. Data used in publications
44,sample2,6,subsection,NaN,None,0.200,False,False,NaN,False,text_not_found,Data management plans should describe how data used in publications resulting from the proposed research will be mad...,None
45,sample2,7,content,NaN,None,0.697,False,False,NaN,False,text_not_found,Research conducted within the Center will be published in appropriate scientific journals. The publisher may restric...,None


## Step 11 — Per-label accuracy

This answers: How well did Llama detect `document_title`, `section`, `subsection`, and `content` separately?


In [15]:
if not block_level_report.empty:
    per_label_accuracy = (
        block_level_report
        .groupby('ref_label')
        .agg(
            total_blocks=('ref_label', 'count'),
            text_found=('similar_text_found', 'sum'),
            label_correct=('label_correct', 'sum'),
            order_correct=('order_correct', 'sum')
        )
        .reset_index()
    )

    per_label_accuracy['text_match_rate'] = (per_label_accuracy['text_found'] / per_label_accuracy['total_blocks']).round(3)
    per_label_accuracy['label_accuracy'] = (per_label_accuracy['label_correct'] / per_label_accuracy['total_blocks']).round(3)
    per_label_accuracy['order_accuracy'] = (per_label_accuracy['order_correct'] / per_label_accuracy['total_blocks']).round(3)
else:
    per_label_accuracy = pd.DataFrame()

per_label_accuracy


,ref_label,total_blocks,text_found,label_correct,order_correct,text_match_rate,label_accuracy,order_accuracy
0,content,49,36,36,28,0.735,0.735,0.571
1,document_title,10,9,9,9,0.900,0.900,0.900
2,section,43,37,36,22,0.860,0.837,0.512
3,subsection,15,8,5,8,0.533,0.333,0.533


## Step 12 — Label confusion table

This answers: When Llama gave the wrong label, what did it confuse?

Example: `section → subsection`.


In [16]:
matched_blocks = block_level_report[block_level_report['similar_text_found'] == True].copy()

if not matched_blocks.empty:
    label_confusion = pd.crosstab(
        matched_blocks['ref_label'],
        matched_blocks['llama_label'],
        rownames=['Reference label'],
        colnames=['Llama label']
    )
else:
    label_confusion = pd.DataFrame()

label_confusion


Llama label,content,document_title,section,subsection
Reference label,,,,
content,36,0,0,0
document_title,0,9,0,0
section,0,0,36,1
subsection,0,0,3,5


## Step 13 — Wrong label pairs only

This gives a compact list of mistakes such as `section → subsection`.


In [17]:
wrong_label_pairs = matched_blocks[matched_blocks['ref_label'] != matched_blocks['llama_label']].copy()

if not wrong_label_pairs.empty:
    wrong_label_summary = (
        wrong_label_pairs
        .groupby(['ref_label', 'llama_label'])
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
    )
else:
    wrong_label_summary = pd.DataFrame(columns=['ref_label', 'llama_label', 'count'])

wrong_label_summary


,ref_label,llama_label,count
1,subsection,section,3
0,section,subsection,1


## Step 14 — Extra Llama blocks

These are blocks detected by Llama that did not match any manual/reference block.


In [18]:
extra_llama_report


,sample_id,llama_idx,llama_label,llama_text,issue
0,sample1,25,section,"Protections for privacy, rights, and confidentiality of human research participants:",extra_or_unmatched_llama_block
1,sample1,26,content,All data used in this study was de-identified prior to use.,extra_or_unmatched_llama_block
2,sample2,3,content,Data management plans should describe whether and how data generated in the course of the proposed research will be ...,extra_or_unmatched_llama_block
3,sample2,4,section,"Data Types and Sources. A brief, high-level description of the data to be generated or used through",extra_or_unmatched_llama_block
4,sample2,5,content,the course of the proposed research and which of these are considered Digital Research Data necessary to Validate th...,extra_or_unmatched_llama_block
5,sample2,6,section,"Content and Format. A statement of plans for data and metadata content and format including, where",extra_or_unmatched_llama_block
6,sample2,7,content,"applicable, a description of documentation plans, annotation of relevant software, and the rationale for the selecti...",extra_or_unmatched_llama_block
7,sample2,8,section,to be less than XX GB. Experimental techniques will generate text and Microsoft Office files and,extra_or_unmatched_llama_block
8,sample2,9,content,videos that are expected to be less than XX GB.,extra_or_unmatched_llama_block
9,sample2,10,section,Data Sharing and Data Preservation. A description of the plans for data sharing and preservation.,extra_or_unmatched_llama_block


## Step 15 — Save all reports

In [19]:
summary_path = output_dir / 'summary_report.csv'
block_path = output_dir / 'block_level_report.csv'
problem_path = output_dir / 'problem_blocks_report.csv'
per_label_path = output_dir / 'per_label_accuracy.csv'
confusion_path = output_dir / 'label_confusion_matrix.csv'
wrong_pairs_path = output_dir / 'wrong_label_pairs.csv'
extra_path = output_dir / 'extra_llama_blocks.csv'

summary_report.to_csv(summary_path, index=False)
block_level_report.to_csv(block_path, index=False)
problem_blocks.to_csv(problem_path, index=False)
per_label_accuracy.to_csv(per_label_path, index=False)
label_confusion.to_csv(confusion_path)
wrong_label_summary.to_csv(wrong_pairs_path, index=False)
extra_llama_report.to_csv(extra_path, index=False)

print('Saved reports:')
for p in [summary_path, block_path, problem_path, per_label_path, confusion_path, wrong_pairs_path, extra_path]:
    print(' -', p)


Saved reports:
 - c:\Users\Nahid\dmpbridge\data\block_comparison_reports_simple\summary_report.csv
 - c:\Users\Nahid\dmpbridge\data\block_comparison_reports_simple\block_level_report.csv
 - c:\Users\Nahid\dmpbridge\data\block_comparison_reports_simple\problem_blocks_report.csv
 - c:\Users\Nahid\dmpbridge\data\block_comparison_reports_simple\per_label_accuracy.csv
 - c:\Users\Nahid\dmpbridge\data\block_comparison_reports_simple\label_confusion_matrix.csv
 - c:\Users\Nahid\dmpbridge\data\block_comparison_reports_simple\wrong_label_pairs.csv
 - c:\Users\Nahid\dmpbridge\data\block_comparison_reports_simple\extra_llama_blocks.csv


## Step 16 — Simple interpretation guide

Use this wording in your results:

- **Text Match Rate** = how many manual blocks were found by Llama.
- **Label Accuracy** = how many manual blocks were found with the correct label.
- **Order Accuracy** = how many manual blocks were found in the correct/near-correct document order.
- **Label Confusion Table** = shows which labels Llama confused, such as `section → subsection`.

For each sample, your main conclusion should be:

> Llama found the text well / not well, labeled the blocks correctly / incorrectly, and preserved / did not preserve document order.
